# Lab 3: Calidad de datos y feature engineering distribuido
**Análisis de Big Data · Magíster en Data Science UDD · Sesión 3 (vie 21 ago)**

**Nombre: Ricardo Castro Vera**

**Objetivos:** convertir las seis dimensiones de calidad en **reglas ejecutables** que devuelven un conteo, producir el **reporte de calidad con columna de decisión** y limpiar con criterio (filtrar, imputar con flag, deduplicar por ventana).

**Entorno:** Google Colab (ejecutar el setup) o Databricks CE (saltar el `pip install` de pyspark).

**Dataset:** NYC Yellow Taxi entre **enero y marzo 2023** (~9,5M viajes, 3 archivos Parquet, ~170 MB). Tres meses, no doce, para que el laboratorio corra en el bloque de clase.

**Entrega:** notebook ejecutado completamente (Kernel → Restart & Run All antes de subir). **Cuenta para el 25% de laboratorios.**

**Rúbrica:** corre completamente ✓ · reporte de calidad con **decisión justificada** por regla ✓ · ninguna regla sobre el umbral filtrada sin justificación escrita (sección 3.4) ✓ · shuffle identificado en el `EXPLAIN` ✓ · uso de IA declarado ✓

> **v2 · Actualizaciones tras la sesión 3.** (a) Nueva sección **0c**: qué ocurre realmente cuando se castea un texto a `double`, pregunta que quedó abierta en clase. (b) Sección **3** reescrita: el filtrado ahora **respeta el umbral visto en clases**.

> Continuidad con el Lab 2: mismo dataset y mismo motor. Allí la pregunta era *cuál motor conviene*, ahora es *si los datos que ese motor lee son confiables y cómo se convierten en variables*.

## 0. Setup

In [1]:
# Colab / Jupyter local (en Databricks CE, no reinstalar pyspark)
%pip install -q pyspark==3.5.4 pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.


In [2]:
import urllib.request, os, glob
MESES = 3                      # ene-mar 2023. Subirlo exige ajustar PERIODO_FIN más abajo.
PERIODO_INI, PERIODO_FIN = "2023-01-01", "2023-04-01"
CORTE_TEST = "2023-03-01"      # train: ene-feb · test: mar

for m in range(1, MESES + 1):
    f = f"yellow_tripdata_2023-{m:02d}.parquet"
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"https://d37ci6vzurychx.cloudfront.net/trip-data/{f}", f)
        print("ok", f)
ARCHIVOS = sorted(glob.glob("yellow_tripdata_2023-*.parquet"))
print(len(ARCHIVOS), "archivos ·", round(sum(os.path.getsize(a) for a in ARCHIVOS)/1e6), "MB")


ok yellow_tripdata_2023-01.parquet
ok yellow_tripdata_2023-02.parquet
ok yellow_tripdata_2023-03.parquet
3 archivos · 152 MB


In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from functools import reduce
import pandas as pd, time

spark = (SparkSession.builder.appName("ADBD-Lab3")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
pd.set_option("display.width", 160)


### 0b. Carga con esquema declarado: la primera regla de calidad

Hallazgo del Lab 2: los Parquet de TLC **cambian tipos y nombres de columna entre meses**. Leer los archivos juntos y confiar en la inferencia es la primera falla de calidad, y ocurre antes de mirar un solo dato.

El patrón correcto es el de abajo: **esquema esperado declarado en el código**, cada archivo leído con el suyo, cast explícito. Esto es la dimensión **Esquema** que se verifica en la sección 2.

In [4]:
# Esquema esperado: nombre -> tipo. Es un contrato, no una sugerencia.
COLS = {
    "VendorID": "bigint",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "RatecodeID": "double",
    "PULocationID": "bigint",
    "DOLocationID": "bigint",
    "payment_type": "bigint",
    "fare_amount": "double",
    "tip_amount": "double",
    "total_amount": "double",
}

partes = [spark.read.parquet(a).select([F.col(c).cast(t).alias(c) for c, t in COLS.items()])
          for a in ARCHIVOS]
bronze = reduce(lambda a, b: a.unionByName(b), partes).cache()   # capa Bronze: crudo, sin juicio
n_bronze = bronze.count()
print(f"Bronze: {n_bronze:,} filas · {len(bronze.columns)} columnas")


Bronze: 9,384,487 filas · 12 columnas


### 0c. ¿Y si el cast no puede? Pregunta abierta en la sesión 3

> **Si se declara `double` y la columna venía como texto, ¿el cast falla?**

**No falla: devuelve `NULL`, en silencio,** que es el peor comportamiento posible, pues suele detectarse cuando el modelo ya está entrenando con muchos datos nulos.

In [5]:
from pyspark.sql.types import StructType, StructField, StringType

crudo = spark.createDataFrame(
    [("12.5",), ("0",), ("3,7",), ("N/A",), ("",), (None,), ("1e3",), (" 8.0 ",), ("$15",), ("9999999999999999999",)],
    StructType([StructField("monto_txt", StringType())]))

(crudo.withColumn("a_double",   F.col("monto_txt").cast("double"))
      .withColumn("try_cast",   F.expr("try_cast(monto_txt AS double)"))
      .withColumn("a_int",      F.col("monto_txt").cast("int"))
      .show(truncate=False))


+-------------------+--------+--------+-----+
|monto_txt          |a_double|try_cast|a_int|
+-------------------+--------+--------+-----+
|12.5               |12.5    |12.5    |12   |
|0                  |0.0     |0.0     |0    |
|3,7                |NULL    |NULL    |NULL |
|N/A                |NULL    |NULL    |NULL |
|                   |NULL    |NULL    |NULL |
|NULL               |NULL    |NULL    |NULL |
|1e3                |1000.0  |1000.0  |NULL |
| 8.0               |8.0     |8.0     |8    |
|$15                |NULL    |NULL    |NULL |
|9999999999999999999|1.0E19  |1.0E19  |NULL |
+-------------------+--------+--------+-----+



**Fila por fila, lo que acaba de pasar:**

| Entrada | `cast("double")` | Por qué importa |
|---|---|---|
| `"12.5"`, `"1e3"`, `" 8.0 "` | 12.5 · 1000.0 · 8.0 | Parsea notación científica y tolera espacios. |
| `"3,7"` | **null** | **Coma decimal.** Un CSV exportado con configuración regional chilena pierde *toda* la columna, sin avisar el error. |
| `"N/A"`, `""`, `"$15"` | **null** | El centinela textual, el vacío y el símbolo de moneda son indistinguibles de un dato ausente después del cast. |
| `"9999999999999999999"` | 1.0E19 en `double`, **null** en `int` | El desborde tampoco avisa: cambia según el tipo destino. |

Tres consecuencias prácticas:

1. **El nulo de la sección 1 puede no venir del origen.** Puede haberlo fabricado la línea de carga. Son dos problemas distintos y se arreglan distinto.
2. `try_cast` devuelve `NULL` **por defecto** al no poder hacer el *casting*, por lo que quien lea el código sabe que la generación de **null** es un comportamento deseado. `cast` a secas, en cambio, no distingue entre una decisión y un descuido.
3. La conversión de `double` a entero **trunca, no redondea** (`8.9 → 8`), y el desborde vuelve a dar `NULL`.

In [6]:
# Modo ANSI: el MISMO cast lanza excepción en vez de devolver null.
# Opcional hasta Spark 3.x, pero pasa a ser el DEFAULT en Spark 4.0. Por lo
# tanto, el mismo notebook cambia de comportamiento al migrar de versión, sin
# tocar una línea.
spark.conf.set("spark.sql.ansi.enabled", True)
try:
    crudo.select(F.col("monto_txt").cast("double")).collect()
    print("sin error")
except Exception as e:
    print("ANSI lanza ->", type(e).__name__, "|", str(e).split("\n")[0][:150])
finally:
    spark.conf.set("spark.sql.ansi.enabled", False)


ANSI lanza -> Py4JJavaError | An error occurred while calling o187.collectToPython.


Con ANSI activado, el comportamiento pasa a ser el descrito en clases. Existen dos opciones: **fallar de forma temprana y ruidosa**, o **seguir y perder datos**. Es responsabilidad nuestra saber en cuál de los dos modos corre el pipeline.

In [7]:
# La regla operativa: un cast es una transformación y, como toda transformación, se audita.
# Nulos ANTES vs DESPUÉS: la diferencia son filas que destruyó la línea de carga.
def auditar_cast(path, cols):
    raw = spark.read.parquet(path)
    presentes = [c for c in cols if c in raw.columns]
    nulos = lambda df: df.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in presentes]).collect()[0].asDict()
    antes = nulos(raw)
    despues = nulos(raw.select([F.col(c).cast(cols[c]).alias(c) for c in presentes]))
    return pd.DataFrame([(c, raw.schema[c].dataType.simpleString(), cols[c],
                          antes[c], despues[c], despues[c] - antes[c]) for c in presentes],
                        columns=["columna", "tipo_origen", "tipo_declarado",
                                 "nulos_antes", "nulos_despues", "creados_por_el_cast"])

audit = auditar_cast(ARCHIVOS[0], COLS)
NULOS_POR_CAST = int(audit["creados_por_el_cast"].sum())
print("Nulos fabricados por el cast:", NULOS_POR_CAST)
audit


Nulos fabricados por el cast: 0


,columna,tipo_origen,tipo_declarado,nulos_antes,nulos_despues,creados_por_el_cast
0,VendorID,bigint,bigint,0,0,0
1,tpep_pickup_datetime,timestamp_ntz,timestamp,0,0,0
2,tpep_dropoff_datetime,timestamp_ntz,timestamp,0,0,0
3,passenger_count,double,double,71743,71743,0
4,trip_distance,double,double,0,0,0
5,RatecodeID,double,double,71743,71743,0
6,PULocationID,bigint,bigint,0,0,0
7,DOLocationID,bigint,bigint,0,0,0
8,payment_type,bigint,bigint,0,0,0
9,fare_amount,double,double,0,0,0


**Leer la columna `creados_por_el_cast`:** en este caso es cero, ya que los tipos de las columnas coinciden y el *casting* no destruye nada. Ahora no existe un supuesto, sino **la certeza** de que ocurre lo esperado.

`NULOS_POR_CAST` entra al reporte de calidad de la sección 2 como una regla más, bajo la dimensión **Esquema**. Es la única regla que mide los datos anulados (o "el daño causado") por el propio pipeline.

## 1. Etapa 1: Perfilado en una sola pasada

Nulos, cardinalidades y rangos. La restricción que define el ejercicio: **una sola pasada sobre los datos**. Un <code>count()</code> por columna sobre 9,5M filas son doce recorridos completos del dataset. El mismo resultado se obtiene con una sola agregación de doce expresiones.

In [8]:
# Nulos y cardinalidad aproximada: dos pasadas, no veinticuatro.
nulos = bronze.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in bronze.columns]).collect()[0].asDict()
card  = bronze.select([F.approx_count_distinct(c).alias(c) for c in bronze.columns]).collect()[0].asDict()

perfil = pd.DataFrame({"columna": list(COLS), "tipo": list(COLS.values())})
perfil["nulos"] = perfil["columna"].map(nulos)
perfil["%_nulos"] = (perfil["nulos"] / n_bronze * 100).round(2)
perfil["card_aprox"] = perfil["columna"].map(card)
perfil


,columna,tipo,nulos,%_nulos,card_aprox
0,VendorID,bigint,0,0.00,3
1,tpep_pickup_datetime,timestamp,0,0.00,4734247
2,tpep_dropoff_datetime,timestamp,0,0.00,4888717
3,passenger_count,double,236179,2.52,10
4,trip_distance,double,0,0.00,5796
5,RatecodeID,double,236179,2.52,7
6,PULocationID,bigint,0,0.00,264
7,DOLocationID,bigint,0,0.00,264
8,payment_type,bigint,0,0.00,6
9,fare_amount,double,0,0.00,8715


In [9]:
# Rangos: el mínimo y el máximo son donde vive la basura.
NUMERICAS = [c for c, t in COLS.items() if t == "double" or c.endswith("ID")]
bronze.select(NUMERICAS).summary("min", "25%", "50%", "75%", "max").toPandas()


,summary,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,fare_amount,tip_amount,total_amount
0,min,1,0.0,0.0,1.0,1,1,-959.9,-96.22,-982.95
1,25%,1,1.0,1.06,1.0,132,114,8.6,1.0,15.5
2,50%,2,1.0,1.79,1.0,162,162,12.8,2.8,20.3
3,75%,2,1.0,3.33,1.0,234,234,20.5,4.25,29.04
4,max,6,9.0,335004.33,99.0,265,265,2203.1,984.3,2208.1


**Lectura del perfilado, responder en esta celda (editarla):**

1.  **¿Qué columnas tienen nulos y en qué proporción? ¿Alguna supera el 5%?**
    Las columnas `passenger_count` y `RatecodeID` tienen nulos, ambas en una proporción del 2.52% (236,179 filas). Ninguna de ellas supera el 5%.
2.  **`min(total_amount)` y `min(trip_distance)`: ¿qué significan esos valores en el negocio del taxi?**
    `min(total_amount)` (-982.95) es un valor anómalo (negativo) que indica un posible error de registro o un ajuste/reembolso inusual. `min(trip_distance)` (0.0) indica viajes de distancia cero, lo cual podría ser una cancelación o un registro defectuoso donde el taxi no se movió.
3.  **`max(trip_distance)`: ¿es un viaje o un error de medición? ¿Con qué criterio se decide?**
    `max(trip_distance)` (335,004.33 millas) es una distancia extremadamente alta para un taxi en NYC y claramente un error de medición. Se decide con el criterio de la plausibilidad física y el contexto geográfico de los viajes en taxi.
4.  **La cardinalidad de `PULocationID` debería ser ~265 (zonas de TLC). ¿Coincide?**
    Sí, la cardinalidad aproximada para `PULocationID` es 264, lo cual coincide con las ~265 zonas de TLC esperadas.

> A escala no se inspecciona a ojo: se declara una regla y se mide cuántas filas no la cumplen. Eso es la etapa 2.

*   Elemento de la lista
*   Elemento de la listacell_id: G2UPyIQEW74R

## 2. Etapa 2: Diccionario de reglas y reporte de calidad

Dos columnas de apoyo primero (<code>duracion_min</code> y <code>velocidad_mph</code>): sin ellas, las reglas de consistencia no se pueden escribir. Son parte del **diagnóstico**, no del set de features.

In [10]:
diag = (bronze
    .withColumn("duracion_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
    .withColumn("velocidad_mph",
        F.when(F.col("duracion_min") > 0, F.col("trip_distance") / (F.col("duracion_min") / 60.0))))


In [11]:
# Diccionario de reglas: nombre -> (dimensión, condición que debe cumplirse, decisión propuesta)
REGLAS = {
    "completitud_pasajeros":      ("Completitud",  F.col("passenger_count").isNotNull(),                        "imputar + flag"),
    "completitud_zona_origen":    ("Completitud",  F.col("PULocationID").isNotNull(),                           "eliminar"),
    "validez_monto":              ("Validez",      F.col("total_amount").between(0.01, 5000),                   "eliminar"),
    "validez_distancia":          ("Validez",      F.col("trip_distance").between(0.01, 200),                   "eliminar"),
    "validez_medio_pago":         ("Validez",      F.col("payment_type").isin([1, 2, 3, 4, 5, 6]),              "eliminar"),
    "consistencia_orden_tiempos": ("Consistencia", F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"), "eliminar"),
    "consistencia_duracion":      ("Consistencia", F.col("duracion_min").between(1, 360),                       "eliminar"),
    "plausibilidad_velocidad":    ("Consistencia", F.col("velocidad_mph") <= 100,                               "eliminar"),
    "actualidad_periodo":         ("Actualidad",   (F.col("tpep_pickup_datetime") >= PERIODO_INI) &
                                                   (F.col("tpep_pickup_datetime") <  PERIODO_FIN),              "eliminar"),
}

# Una sola pasada para las nueve reglas.
# Ojo con los nulos: 'when(cond, 0).otherwise(1)' cuenta el nulo como FALLA (una condición
# nula no es una condición cumplida). Con sum(~cond) el nulo se ignoraría y el reporte mentiría.
exprs = [F.sum(F.when(cond, 0).otherwise(1)).alias(nombre) for nombre, (_, cond, _) in REGLAS.items()]
fallos = diag.agg(*exprs).collect()[0].asDict()


In [12]:
# Unicidad: se mide aparte porque cuesta un shuffle (countDistinct sobre la clave de negocio).
CLAVE = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]
n_distintas = diag.select(CLAVE).distinct().count()
fallos["unicidad_clave_viaje"] = n_bronze - n_distintas

# Esquema: contrato declarado vs. observado.
observado = {f.name: f.dataType.simpleString() for f in bronze.schema.fields}
desvios = {c: (t, observado.get(c)) for c, t in COLS.items() if observado.get(c) != t}
print("Desvíos de esquema:", desvios if desvios else "ninguno (el cast de la carga los absorbió)")


Desvíos de esquema: ninguno (el cast de la carga los absorbió)


In [13]:
filas = [(dim, nombre, fallos[nombre], round(fallos[nombre] / n_bronze * 100, 2), dec)
         for nombre, (dim, _, dec) in REGLAS.items()]
filas.append(("Unicidad", "unicidad_clave_viaje", fallos["unicidad_clave_viaje"],
              round(fallos["unicidad_clave_viaje"] / n_bronze * 100, 2), "deduplicar por ventana"))
# Sección 0c: nulos que fabricó el pipeline de carga, no el origen.
filas.append(("Esquema", "cast_sin_perdida", NULOS_POR_CAST,
              round(NULOS_POR_CAST / n_bronze * 100, 2),
              "revisar la declaración de tipos" if NULOS_POR_CAST else "sin acción"))

reporte = pd.DataFrame(filas, columns=["dimensión", "regla", "filas_fallidas", "%", "decisión"])
reporte = reporte.sort_values("%", ascending=False).reset_index(drop=True)
reporte


,dimensión,regla,filas_fallidas,%,decisión
0,Completitud,completitud_pasajeros,236179,2.52,imputar + flag
1,Validez,validez_medio_pago,236179,2.52,eliminar
2,Validez,validez_distancia,135731,1.45,eliminar
3,Consistencia,consistencia_duracion,112340,1.20,eliminar
4,Validez,validez_monto,81601,0.87,eliminar
5,Unicidad,unicidad_clave_viaje,70121,0.75,deduplicar por ventana
6,Consistencia,plausibilidad_velocidad,9033,0.10,eliminar
7,Consistencia,consistencia_orden_tiempos,3839,0.04,eliminar
8,Completitud,completitud_zona_origen,0,0.00,eliminar
9,Actualidad,actualidad_periodo,139,0.00,eliminar


### 2b. El reporte es el entregable. Se debe completar la justificación.

La columna `decisión` viene propuesta, y es **la que se evalúa**. Criterio de la clase: bajo el **1%** se filtra sin discusión, sobre el **5%** hay que entender qué pasa *antes* de tocar nada.

**Responder en esta celda (editarla):**

1.  **¿Qué regla tiene el mayor porcentaje de fallas y cuál es la causa?**
    Las reglas `completitud_pasajeros` y `validez_medio_pago` comparten el mayor porcentaje de fallas (2.52%). En el caso de `completitud_pasajeros`, la causa son valores nulos en la columna. Para `validez_medio_pago`, la causa son valores fuera del rango esperado, y parece estar correlacionada con los nulos de `passenger_count`.
2.  **¿Alguna decisión propuesta *a priori* cambia a la luz de los porcentajes reales? ¿Cuáles y por qué?**
    Sí, las decisiones para `validez_distancia` (1.45%), `validez_medio_pago` (2.52%) y `consistencia_duracion` (1.20%) podrían cambiar de `eliminar` a `conservar_con_flag`. Esto se debe a que sus porcentajes de falla están entre el 1% y el 5%, lo que sugiere que podría haber segmentos de negocio legítimos (aunque atípicos) que se perderían al filtrar, y mantenerlos con un flag permitiría al modelo aprender de ellos.
3.  **`completitud_pasajeros` se resuelve con `imputar + flag` y no con eliminar. ¿Qué se perdería al eliminar esas filas?**
    Al eliminar esas filas, se perdería el 2.52% de los datos (236,179 filas). Esta es una cantidad significativa de información que, si se puede imputar y señalar con un flag, permite conservar la representatividad del dataset para el entrenamiento del modelo.
4.  **Una regla que no está en el diccionario y que este dataset necesite:**
    Una regla para la `validez_fare_amount` (`fare_amount` entre 0.01 y un umbral razonable), ya que el perfilado mostró valores mínimos negativos que no son lógicos para una tarifa. Esto abordaría la dimensión de **Validez**.

> Este cuadro, con esta columna de decisión, es una de las secciones del informe de la **Fase 1** (vie 28 ago).

## 3. Etapa 3: Limpieza: filtrar, marcar, deduplicar

Aquí se ejecuta la columna `decisión` de la sección 2b. Y la primera decisión es **cuáles reglas tienen permiso para descartar filas**: el criterio de la clase (≤1% se filtra sin discusión, >5% no se toca hasta entender) tiene que estar **en el código**, no solo en la lámina. Un pipeline que descarta el 8% de las filas sin que nadie lo note es exactamente lo que la sección 2 intenta prevenir.

In [14]:
# 3.1 La política de la sección 2b, ejecutable:
#   <= 1%   -> se filtra sin discusión
#   1% a 5% -> se filtra SOLO con justificación escrita
#   > 5%    -> no se filtra hasta entender la causa
UMBRAL_LIBRE, UMBRAL_ALERTA = 1.0, 5.0
pct = {r: fallos[r] / n_bronze * 100 for r in REGLAS}

# 'completitud_pasajeros' queda fuera: su decisión fue imputar, no descartar.
REGLAS_FILTRO = {n: v for n, v in REGLAS.items() if n != "completitud_pasajeros"}
automaticas  = [r for r in REGLAS_FILTRO if pct[r] <= UMBRAL_LIBRE]
a_justificar = [r for r in REGLAS_FILTRO if pct[r] >  UMBRAL_LIBRE]

print(f"Filtrado automático — {len(automaticas)} reglas bajo el {UMBRAL_LIBRE}%:")
for r in automaticas:
    print(f"    {r:<30}{pct[r]:>7.2f}%")
print(f"\nRequieren decisión explícita — {len(a_justificar)} reglas:")
for r in a_justificar:
    alerta = "  <-- SOBRE EL 5%: entender antes de tocar" if pct[r] > UMBRAL_ALERTA else ""
    print(f"    {r:<30}{pct[r]:>7.2f}%{alerta}")


Filtrado automático — 5 reglas bajo el 1.0%:
    completitud_zona_origen          0.00%
    validez_monto                    0.87%
    consistencia_orden_tiempos       0.04%
    plausibilidad_velocidad          0.10%
    actualidad_periodo               0.00%

Requieren decisión explícita — 3 reglas:
    validez_distancia                1.45%
    validez_medio_pago               2.52%
    consistencia_duracion            1.20%


In [15]:
# 3.2 Se filtra SOLO con las reglas que están bajo el umbral. Las demás siguen adentro.
cond_auto = (reduce(lambda a, b: a & b, [F.coalesce(REGLAS[r][1], F.lit(False)) for r in automaticas])
             if automaticas else F.lit(True))
silver = diag.filter(cond_auto)
n_auto = silver.count()
print(f"Bronze {n_bronze:,} -> tras filtrado automático {n_auto:,} ({(1 - n_auto/n_bronze)*100:.2f}% descartado)")


Bronze 9,384,487 -> tras filtrado automático 9,294,106 (0.96% descartado)


In [16]:
# 3.3 Evidencia antes de decidir: qué hay dentro de lo que cada regla pendiente quiere descartar.
for r in a_justificar:
    print(f"\n=== {r} · {pct[r]:.2f}% de las filas ===")
    (silver.filter(~F.coalesce(REGLAS[r][1], F.lit(False)))
           .select("trip_distance", "duracion_min", "velocidad_mph", "total_amount", "passenger_count")
           .summary("count", "min", "50%", "max").show())



=== validez_distancia · 1.45% de las filas ===
+-------+-------------+--------------------+-----------------+------------+---------------+
|summary|trip_distance|        duracion_min|    velocidad_mph|total_amount|passenger_count|
+-------+-------------+--------------------+-----------------+------------+---------------+
|  count|       121596|              121596|           121596|      121596|         102849|
|    min|          0.0|0.016666666666666666|              0.0|        0.01|            0.0|
|    50%|          0.0|                 1.8|              0.0|       20.02|            1.0|
|    max|        204.1|  10029.183333333332|61.60992788864665|      1000.0|            9.0|
+-------+-------------+--------------------+-----------------+------------+---------------+


=== validez_medio_pago · 2.52% de las filas ===
+-------+-------------+--------------------+------------------+------------+---------------+
|summary|trip_distance|        duracion_min|     velocidad_mph|total_amou

**Responder en esta celda (editarla):**

Al observar la mediana y los extremos de las reglas pendientes (`validez_distancia`, `validez_medio_pago`, `consistencia_duracion`):

*   **`validez_distancia`:** La mediana en 0.0 sugiere muchos viajes cancelados o muy cortos, que son datos potencialmente rotos. Sin embargo, el valor máximo de 204.1 millas, aunque alto, podría representar un segmento legítimo (ej. viajes largos a aeropuertos), por lo que se prefiere `conservar_con_flag`.
*   **`validez_medio_pago`:** La ausencia completa de `passenger_count` para estos registros, además de un `payment_type` inválido, sugiere que son datos completamente rotos o registros incompletos que no aportan valor y deberían ser filtrados.
*   **`consistencia_duracion`:** Los valores mínimos de ~1 segundo y máximos de más de 10,000 minutos (más de 6 días) son claramente errores de medición o de operación del taxímetro. Estos son datos rotos y deberían ser filtrados.

### 3.3b Decisión por regla

Cada regla pendiente necesita **una acción y una justificación escrita**. Las dos acciones posibles:

- **`filtrar`**: las filas se descartan. Exige justificación: qué son esas filas y por qué no pertenecen al análisis.
- **`conservar_con_flag`**: las filas se quedan con una columna `incumple_<regla>` que las marca. Es la opción correcta cuando lo que la regla llama "error" resulta ser un segmento real del negocio: el modelo puede usar la marca como señal.

Editar el diccionario de la celda siguiente. Mientras una justificación diga `PENDIENTE`, el notebook lo reporta y **la entrega está incompleta**.



## 3.4 Editar: una entrada por cada regla listada en 3.1 como pendiente.



In [30]:
DECISIONES = {
    "validez_distancia": (
        "conservar_con_flag",
        "Muchos viajes con distancia 0.0 indican errores o cancelaciones. Si bien algunos viajes muy largos podrían ser legítimos (ej. a aeropuertos), se prefiere flaggear estos casos para que el modelo pueda aprender de ellos."
    ),
    "validez_medio_pago": (
        "filtrar",
        "Los registros que fallan esta regla también presentan un conteo de pasajeros nulo y tipos de pago inválidos, lo que sugiere datos severamente corruptos e inútiles para el análisis."
    ),
    "consistencia_duracion": (
        "filtrar",
        "Los viajes con duración menor a 1 minuto son probablemente cancelaciones o errores de taxímetro. Los viajes con duración superior a 6 horas son casi con certeza errores de registro. Ninguno representa un servicio de taxi válido."
    )
}

# Formato esperado — reemplazar por las decisiones del equipo:
# DECISIONES["consistencia_duracion"] = (
#     "filtrar",
#     "Viajes bajo 1 minuto son cancelaciones con el taxímetro ya abierto: no son viajes. "
#     "Los sobre 6 horas son taxímetros que no se cerraron. Ninguno de los dos casos aporta al objetivo.")

for r in a_justificar:
    assert r in DECISIONES, f"falta la decisión de {r}"
for r, (accion, just) in DECISIONES.items():
    assert accion in ("filtrar", "conservar_con_flag"), f"acción inválida en {r}: {accion}"
    if just.startswith("PENDIENTE"):
        print(f"[!] {r} ({pct[r]:.2f}%) sin justificación — la entrega no está completa")

## 3.5 Aplicación de las decisiones de calidad de datos.
Esta sección ejecuta las acciones definidas en el diccionario `DECISIONES` para las reglas que requerían una justificación explícita (`a_justificar`). El objetivo es refinar el DataFrame `silver` ya sea descartando filas o añadiendo flags.

In [18]:
for r, (accion, _) in DECISIONES.items():
    # Recupera la condición Spark de la regla y asegura que los NULLs se traten como FALLOS (False).
    # Esto es crucial para que `filter` y `~cond` funcionen correctamente en presencia de valores ausentes.
    cond = F.coalesce(REGLAS[r][1], F.lit(False))
    if accion == "filtrar":
        # Si la decisión es 'filtrar', se eliminan las filas que no cumplen la condición de la regla.
        antes = silver.count()
        silver = silver.filter(cond)
        print(f"filtrar             {r:<30} -{antes - silver.count():>10,} filas")
    else:
        # Si la decisión es 'conservar_con_flag', las filas que no cumplen la regla se mantienen,
        # pero se les añade una nueva columna (un flag binario) que indica su incumplimiento.
        # Esto permite que un modelo aprenda de estas "anomalías" sin descartar los datos.
        silver = silver.withColumn(f"incumple_{r}", (~cond).cast("int"))
        print(f"conservar_con_flag  {r:<30}  columna incumple_{r}")

# Reporte final del estado del DataFrame `silver` después de aplicar todas las decisiones (automáticas y justificadas).
n_silver = silver.count()
print(f"\nSilver: {n_silver:,} filas · {(1 - n_silver/n_bronze)*100:.2f}% descartado desde Bronze")

conservar_con_flag  validez_distancia               columna incumple_validez_distancia
conservar_con_flag  validez_medio_pago              columna incumple_validez_medio_pago
conservar_con_flag  consistencia_duracion           columna incumple_consistencia_duracion

Silver: 9,294,106 filas · 0.96% descartado desde Bronze


## 3.6 Gestión de Nulos para `passenger_count`: Marcado de ausencia de valor vs. Imputación.
Aquí se crea una nueva columna `pasajeros_faltante` (un 'flag') para indicar si el `passenger_count` original era nulo. Dado que esta es una operación determinista (se evalúa fila a fila y no 'aprende' nada del conjunto de datos), puede realizarse en esta etapa sin riesgo de data leakage.

In [19]:
silver = silver.withColumn("pasajeros_faltante", F.col("passenger_count").isNull().cast("int"))
silver.groupBy("pasajeros_faltante").count().show()

# La imputación (por ejemplo, reemplazar nulos con la mediana) es una operación que 'aprende'
# un parámetro de los datos (la mediana). Para evitar la fuga de información (data leakage),
# esta acción se pospone al pipeline de la sección 6, donde se ajustará **exclusivamente**
# con el conjunto de entrenamiento (`train`).

+------------------+-------+
|pasajeros_faltante|  count|
+------------------+-------+
|                 1| 234511|
|                 0|9059595|
+------------------+-------+



## 3.7 Deduplicación: explorando las tres estrategias principales y sus impactos.
Esta sección compara diferentes métodos para eliminar filas duplicadas, midiendo su rendimiento y analizando las implicaciones de cada uno en términos de determinismo y audibilidad.

In [20]:


def cronometrar(etiqueta, fn):
    # Función de utilidad para medir el tiempo de ejecución y contar las filas resultantes.
    t0 = time.perf_counter(); n = fn(); print(f"{etiqueta:<34} {n:>12,} filas · {time.perf_counter()-t0:6.2f} s")
    return n

# --- Estrategia 1: Filas idénticas en TODAS las columnas (duplicado "exacto") ---
# `dropDuplicates()` sin argumentos considera que dos filas son duplicadas si todos sus valores
# en todas las columnas coinciden. Esta operación es computacionalmente intensiva (requiere un
# shuffle completo de todo el DataFrame) y, semánticamente, retiene una copia arbitraria de los
# duplicados, lo que la hace no determinista y difícil de auditar.
cronometrar("dropDuplicates() [todas cols]", lambda: silver.dropDuplicates().count())

# --- Estrategia 2: Duplicados por clave de negocio ---
# `dropDuplicates(CLAVE)` define duplicados basándose únicamente en las columnas que forman
# la clave de negocio (`CLAVE`). Si varias filas comparten la misma clave, se conserva una copia
# de forma arbitraria (no determinista). Aunque reduce el alcance del shuffle solo a las columnas
# de la clave, la arbitrariedad en la selección sigue siendo una desventaja en muchos escenarios.
cronometrar("dropDuplicates(CLAVE)", lambda: silver.dropDuplicates(CLAVE).count())

# --- Estrategia 3: Deduplicación determinista usando funciones de ventana (Estrategia Preferida) ---
# Este método utiliza una función de ventana para asignar un número de fila (`row_number`) a cada
# registro dentro de particiones definidas por la clave de negocio (`CLAVE`), y las ordena según
# un criterio específico (aquí, `total_amount` y `fare_amount` descendente). Esto permite
# seleccionar determinísticamente qué copia conservar (ej. la de mayor monto total).
# Es auditable y su comportamiento es predecible, lo que la convierte en la opción ideal para
# la calidad de datos.
w_dup = Window.partitionBy(*CLAVE).orderBy(F.col("total_amount").desc(), F.col("fare_amount").desc())
dedup = silver.withColumn("rn", F.row_number().over(w_dup)).filter("rn = 1").drop("rn")
n_dedup = cronometrar("ventana row_number (elegida)", lambda: dedup.count())


dropDuplicates() [todas cols]         9,294,106 filas ·  66.00 s
dropDuplicates(CLAVE)                 9,294,088 filas ·  39.92 s
ventana row_number (elegida)          9,294,088 filas ·  40.82 s


**Sobre la Estrategia 3:** el orden es `total_amount DESC`, de dos registros con la misma clave de viaje se conserva el de monto mayor, asumiendo que la corrección tarifaria llegó después. En un pipeline con columna de versión (`actualizado`, `ingest_ts`) el orden sería esa columna; TLC no la publica, así que se declara el criterio en su lugar. **Lo que no se puede hacer es dejar que Spark elija.**

**Responder:** ¿por qué la Estrategia 1 tarda más que la 2 y la 3? ¿Qué compara cada una en el shuffle? *(respuesta)*

*   **Estrategia 1 (`dropDuplicates()` - todas las columnas):** Tarda más porque considera **todas las columnas** para identificar duplicados. Esto implica que el shuffle debe enviar y comparar una mayor cantidad de datos a través de la red, lo que es computacionalmente más intensivo.
*   **Estrategia 2 (`dropDuplicates(CLAVE)`):** Tarda menos que la Estrategia 1 porque solo compara las columnas especificadas en `CLAVE` (`VendorID`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `PULocationID`, `DOLocationID`). El shuffle se realiza solo sobre estos campos, reduciendo la cantidad de datos a mover y comparar.
*   **Estrategia 3 (`ventana row_number`):** Tiene un rendimiento comparable a la Estrategia 2. Aunque agrega una operación de `orderBy` dentro de cada partición, el shuffle sigue siendo solo sobre las columnas de la `CLAVE` (`PULocationID` en este caso para la partición), lo que la hace mucho más eficiente que comparar todas las columnas.

## 4. Etapa 4: Tres features: temporal, razón y ventana solo-pasado

**Variable objetivo declarada primero.** Sin objetivo no existe la prueba del reloj: la fuga se define respecto a un instante de predicción.

> **Objetivo:** <code>propina_alta</code>: el viaje deja propina mayor al 20% de la tarifa.
> **Instante de predicción:** fin del viaje, antes de que se registre el pago.

Restricción del dominio: la propina en efectivo **no se registra** en TLC (queda en 0). El universo se limita a <code>payment_type = 1</code> (tarjeta); esa restricción es un sesgo de selección y se declara, no se esconde.

In [21]:
base = (dedup
    .filter(F.col("payment_type") == 1)                      # solo tarjeta: es donde la propina existe
    .withColumn("propina_alta", (F.col("tip_amount") > 0.2 * F.col("fare_amount")).cast("int"))
    .withColumn("fecha", F.to_date("tpep_pickup_datetime")))

base.groupBy("propina_alta").count().show()   # balance de clases: dato para el modelo de la Fase 2


+------------+-------+
|propina_alta|  count|
+------------+-------+
|           1|5538090|
|           0|1869542|
+------------+-------+



In [22]:
# F1 · Derivadas del registro (familia 1): la hora del viaje ya está en la fila.
gold = (base
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .withColumn("dia_semana", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("es_finde", F.col("dia_semana").isin([1, 7]).cast("int"))
    .withColumn("franja", F.when(F.col("hora") < 6,  "madrugada")
                           .when(F.col("hora") < 12, "mañana")
                           .when(F.col("hora") < 19, "tarde")
                           .otherwise("noche")))

# F2 · Razón entre columnas (familia 1): informa más que cada columna por separado.
gold = gold.withColumn("tarifa_por_milla", F.col("fare_amount") / F.col("trip_distance"))
# NOTA: 'total_amount' NO entra al set de features. total = fare + tip + extras, o sea CONTIENE
# el objetivo. Es el caso de fuga más común y el más difícil de ver: la columna parece inocente.


In [23]:
# F3 · Agregada por entidad con ventana (familia 3): demanda de la zona en los 7 días ANTERIORES.
diario = (gold.groupBy(F.col("PULocationID").alias("zona"), "fecha")
              .agg(F.count(F.lit(1)).alias("viajes_dia"))
              .withColumn("dia", F.datediff("fecha", F.lit(PERIODO_INI).cast("date"))))

# rangeBetween(-7, -1) sobre días calendario: la cota superior -1 EXCLUYE el día actual.
# Con rowsBetween(-7, -1) se contarían 7 FILAS, no 7 días: si una zona no tuvo viajes un día,
# la ventana se estiraría hacia atrás sin avisar. Con datos reales siempre faltan días.
w7 = Window.partitionBy("zona").orderBy("dia").rangeBetween(-7, -1)
diario = diario.withColumn("demanda_media_7d", F.round(F.avg("viajes_dia").over(w7), 1))

gold = gold.join(diario.selectExpr("zona as PULocationID", "fecha", "demanda_media_7d"),
                 on=["PULocationID", "fecha"], how="left")

gold.select("PULocationID", "fecha", "hora", "franja", "trip_distance", "duracion_min",
            "velocidad_mph", "tarifa_por_milla", "demanda_media_7d", "propina_alta").show(8)


+------------+----------+----+---------+-------------+------------------+------------------+------------------+----------------+------------+
|PULocationID|     fecha|hora|   franja|trip_distance|      duracion_min|     velocidad_mph|  tarifa_por_milla|demanda_media_7d|propina_alta|
+------------+----------+----+---------+-------------+------------------+------------------+------------------+----------------+------------+
|         164|2023-01-01|   0|madrugada|          0.9| 6.266666666666667| 8.617021276595745| 8.777777777777779|            NULL|           0|
|         142|2023-01-01|   0|madrugada|          3.0|             10.55|17.061611374407583| 4.966666666666667|            NULL|           1|
|         236|2023-01-01|   0|madrugada|          2.8|             17.95|  9.35933147632312| 6.571428571428571|            NULL|           1|
|         158|2023-01-01|   0|madrugada|          0.6| 4.416666666666667| 8.150943396226415| 9.666666666666666|            NULL|           1|
|     

## 5. Etapa 5: <code>EXPLAIN</code> del pipeline: ¿dónde está el shuffle?

Cierre del círculo con el Módulo 1. El pipeline de arriba parece una secuencia de <code>withColumn</code>, pero dos operaciones mueven datos por la red y una tercera pudo evitarlo.

In [24]:
gold.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [PULocationID#44L, fecha#35720, VendorID#38L, tpep_pickup_datetime#39, tpep_dropoff_datetime#40, passenger_count#41, trip_distance#42, RatecodeID#43, DOLocationID#45L, payment_type#46L, fare_amount#47, tip_amount#48, total_amount#49, duracion_min#31035, velocidad_mph#31049, incumple_validez_distancia#33941, incumple_validez_medio_pago#33957, incumple_consistencia_duracion#33974, pasajeros_faltante#34314, propina_alta#35700, hora#36076, dia_semana#36098, es_finde#36121, franja#36145, ... 2 more fields]
   +- SortMergeJoin [PULocationID#44L, fecha#35720], [PULocationID#36240L, fecha#36301], LeftOuter
      :- Sort [PULocationID#44L ASC NULLS FIRST, fecha#35720 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(PULocationID#44L, fecha#35720, 8), ENSURE_REQUIREMENTS, [plan_id=2899]
      :     +- Project [VendorID#38L, tpep_pickup_datetime#39, tpep_dropoff_datetime#40, passenger_count#41, trip_distance#42, Rat

### 5b. Lectura del plan (entregable)

**Responder en esta celda (editarla), con el nombre del operador y la línea del plan:**

1.  **Exchange:** ¿cuántos hay y qué operación provoca cada uno? *(respuesta)*
    Se observan 3 `Exchange` explícitos:
    *   `Exchange hashpartitioning(PULocationID#44L, fecha#35720, 8)`: Provocado por el `SortMergeJoin` del DataFrame `gold` con el DataFrame `diario`. Necesita co-localizar los datos para la unión.
    *   `Exchange hashpartitioning(zona#36240L, fecha#36301, 8)`: También provocado por el `SortMergeJoin`, pero para el DataFrame `diario`.
    *   `Exchange hashpartitioning(zona#36240L, 8)`: Provocado por la función de ventana `Window.partitionBy("zona")` utilizada para calcular `demanda_media_7d` en el DataFrame `diario`.
2.  **La ventana de `demanda_media_7d` particiona por `zona`: ¿qué `Exchange hashpartitioning` le corresponde? *(respuesta)*
    Le corresponde el `Exchange hashpartitioning(zona#36240L, 8), ENSURE_REQUIREMENTS, [plan_id=2904]`, que es el que particiona por la columna `zona` (que es el alias de `PULocationID`).
3.  **El join contra `diario` (~24.000 filas): ¿aparece `BroadcastHashJoin` o `SortMergeJoin`? ¿Qué lo decidió? *(respuesta)*
    Aparece `SortMergeJoin [PULocationID#44L, fecha#35720], [PULocationID#36240L, fecha#36301], LeftOuter`. Spark decidió usar `SortMergeJoin` en lugar de `BroadcastHashJoin`. Esto podría deberse a que el tamaño del DataFrame `diario` (aunque pequeño en número de filas) superó el umbral configurado para `spark.sql.autoBroadcastJoinThreshold`, o a que las estadísticas no estaban disponibles para que AQE tomara una decisión de broadcast dinámicamente.
4.  **`ReadSchema`:** ¿cuántas columnas lee de las 19 del Parquet original, y por qué no las 19? *(respuesta)*
    Lee 12 columnas. No lee las 19 columnas originales debido a la optimización de **column pruning**. Spark solo lee las columnas que son estrictamente necesarias para el procesamiento de la consulta, según lo definido en el `COLS` (`Esquema esperado`) del inicio del notebook y las operaciones subsiguientes.
5.  **¿Qué hizo AQE (buscar `AdaptiveSparkPlan`)? *(respuesta)*
    La presencia de `AdaptiveSparkPlan isFinalPlan=false` indica que Adaptive Query Execution (AQE) está habilitado. AQE optimiza dinámicamente el plan de ejecución de la consulta en tiempo de ejecución, basándose en las características reales de los datos, ajustando estrategias de join, número de particiones, etc. El `isFinalPlan=false` significa que el plan aún podría ser modificado durante la ejecución para una mayor eficiencia.

## 6. Fuga de información: la prueba del reloj y el split temporal

Cada feature se somete a la misma pregunta: **en el instante de la predicción, ¿este valor ya existe?**

| Feature | Familia | ¿Existe al final del viaje? | Veredicto |
|---|---|---|---|
| <code>hora</code>, <code>es_finde</code>, <code>franja</code> | Derivada del registro | Sí, desde que empieza | Limpia |
| <code>trip_distance</code>, <code>duracion_min</code>, <code>velocidad_mph</code> | Derivada del registro | Sí, al bajar el pasajero | Limpia |
| <code>tarifa_por_milla</code> | Razón | Sí, el taxímetro cierra antes del pago | Limpia |
| <code>demanda_media_7d</code> | Agregada por entidad | Sí — la ventana corta en <code>-1</code> | Limpia |
| <code>total_amount</code> | Razón | **Incluye la propina** | **Fuga: excluida** |
| <code>tip_amount</code> | — | Es el objetivo | **Objetivo, no feature** |

La segunda forma de fuga no se ve en la tabla de features, sino en el **orden de las operaciones**: un <code>Imputer</code> o un <code>StandardScaler</code> ajustado sobre todo el dataset filtra información del futuro hacia el train. Por eso el split va **antes** del <code>fit</code>, y el split es **temporal**, no aleatorio: un split aleatorio entrena con marzo para predecir enero.

In [25]:
train = gold.filter(F.col("fecha") <  CORTE_TEST)
test  = gold.filter(F.col("fecha") >= CORTE_TEST)
print(f"train (ene-feb): {train.count():,} · test (mar): {test.count():,}")


train (ene-feb): 4,718,987 · test (mar): 2,688,645


In [26]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, StandardScaler, VectorAssembler

gold = gold.withColumn("ratecode_str", F.col("RatecodeID").cast("string"))
train = train.withColumn("ratecode_str", F.col("RatecodeID").cast("string"))
test  = test.withColumn("ratecode_str",  F.col("RatecodeID").cast("string"))

A_IMPUTAR   = ["passenger_count", "demanda_media_7d", "tarifa_por_milla", "velocidad_mph"]
IMPUTADAS   = [c + "_imp" for c in A_IMPUTAR]
NUM_DIRECTAS = ["trip_distance", "duracion_min", "hora", "es_finde", "pasajeros_faltante"]
CATEGORICAS = ["franja", "ratecode_str"]

etapas = [
    Imputer(strategy="median", inputCols=A_IMPUTAR, outputCols=IMPUTADAS),
    StringIndexer(inputCols=CATEGORICAS, outputCols=[c + "_idx" for c in CATEGORICAS],
                  handleInvalid="keep"),      # 'keep': categoría no vista en train no rompe el test
    OneHotEncoder(inputCols=[c + "_idx" for c in CATEGORICAS],
                  outputCols=[c + "_ohe" for c in CATEGORICAS], handleInvalid="keep"),
    VectorAssembler(inputCols=IMPUTADAS + NUM_DIRECTAS, outputCol="num_vec"),
    StandardScaler(inputCol="num_vec", outputCol="num_esc", withMean=True, withStd=True),
    VectorAssembler(inputCols=["num_esc"] + [c + "_ohe" for c in CATEGORICAS], outputCol="features"),
]

# fit SOLO con train. Si algún VectorAssembler lanza error por un nulo, es buena noticia:
# significa que una fila esquivó las reglas de la sección 2 y el contrato la detuvo.
prep = Pipeline(stages=etapas).fit(train)


In [27]:
# Prueba de que los parámetros salieron solo de train: las medianas aprendidas.
print("Medianas del Imputer (ajustadas con ene-feb):")
prep.stages[0].surrogateDF.show()

print("Media y desviación del StandardScaler — 5 primeras:")
print("  media:", [round(float(x), 3) for x in prep.stages[4].mean.toArray()[:5]])
print("  std  :", [round(float(x), 3) for x in prep.stages[4].std.toArray()[:5]])

# El mismo modelo transforma ambos conjuntos. test NUNCA vuelve a ajustar nada.
train_prep, test_prep = prep.transform(train), prep.transform(test)
train_prep.select("features", "propina_alta").show(3, truncate=80)
print("dimensión del vector de features:", len(train_prep.select("features").head()["features"]))


Medianas del Imputer (ajustadas con ene-feb):
+---------------+----------------+----------------+------------------+
|passenger_count|demanda_media_7d|tarifa_por_milla|     velocidad_mph|
+---------------+----------------+----------------+------------------+
|            1.0|          2115.4|           6.875|10.150375939849624|
+---------------+----------------+----------------+------------------+

Media y desviación del StandardScaler — 5 primeras:
  media: [1.349, 2093.287, 8.715, 11.96, 3.346]
  std  : [0.886, 1030.377, 88.001, 6.744, 4.319]
+--------------------------------------------------------------------------------+------------+
|                                                                        features|propina_alta|
+--------------------------------------------------------------------------------+------------+
|(22,[0,1,2,3,4,5,6,7,12,14],[0.734856614985637,0.021461047693935153,7.1239572...|           0|
|(22,[0,1,2,3,4,5,6,7,12,14],[-0.3939919651171866,0.0214610476939

## 7. Registro de variables: el anexo de la Fase 2

Un feature store en su versión mínima: el conjunto de features es un artefacto **con nombre, definición y versión**, no una celda perdida en un notebook. Sin este cuadro, nadie puede reproducir la tabla Gold.

In [28]:
diccionario = pd.DataFrame([
    ("hora",               "temporal / registro", "Hora de inicio del viaje (0-23)",                        "tpep_pickup_datetime", "—",            "no"),
    ("es_finde",           "temporal / registro", "1 si el viaje inicia sábado o domingo",                  "tpep_pickup_datetime", "—",            "no"),
    ("franja",             "categórica",          "madrugada / mañana / tarde / noche (one-hot)",           "hora",                 "—",            "no"),
    ("ratecode_str",       "categórica",          "Código tarifario TLC (one-hot, handleInvalid=keep)",     "RatecodeID",           "—",            "no"),
    ("trip_distance",      "registro",            "Millas recorridas según taxímetro",                      "trip_distance",        "—",            "no"),
    ("duracion_min",       "razón",               "Minutos entre pickup y dropoff",                         "dropoff - pickup",     "—",            "no"),
    ("velocidad_mph",      "razón",               "trip_distance / (duracion_min / 60)",                    "derivada",             "—",            "no"),
    ("tarifa_por_milla",   "razón",               "fare_amount / trip_distance",                            "derivada",             "—",            "no"),
    ("demanda_media_7d",   "agregada / ventana",  "Viajes diarios promedio de la zona, 7 días previos",     "agregado (zona, fecha)", "[-7d, -1d]",  "no"),
    ("pasajeros_faltante", "flag de calidad",     "1 si passenger_count venía nulo",                        "regla completitud",    "—",            "no"),
    ("passenger_count_imp","registro imputado",   "Pasajeros; mediana aprendida SOLO con train",            "Imputer(median)",      "train",        "no"),
], columns=["feature", "familia", "definición", "origen", "ventana", "riesgo_de_fuga"])

diccionario["version"] = "gold_taxi_v1"
diccionario.to_csv("diccionario_features_v1.csv", index=False)
diccionario


,feature,familia,definición,origen,ventana,riesgo_de_fuga,version
0,hora,temporal / registro,Hora de inicio del viaje (0-23),tpep_pickup_datetime,—,no,gold_taxi_v1
1,es_finde,temporal / registro,1 si el viaje inicia sábado o domingo,tpep_pickup_datetime,—,no,gold_taxi_v1
2,franja,categórica,madrugada / mañana / tarde / noche (one-hot),hora,—,no,gold_taxi_v1
3,ratecode_str,categórica,"Código tarifario TLC (one-hot, handleInvalid=k...",RatecodeID,—,no,gold_taxi_v1
4,trip_distance,registro,Millas recorridas según taxímetro,trip_distance,—,no,gold_taxi_v1
5,duracion_min,razón,Minutos entre pickup y dropoff,dropoff - pickup,—,no,gold_taxi_v1
6,velocidad_mph,razón,trip_distance / (duracion_min / 60),derivada,—,no,gold_taxi_v1
7,tarifa_por_milla,razón,fare_amount / trip_distance,derivada,—,no,gold_taxi_v1
8,demanda_media_7d,agregada / ventana,"Viajes diarios promedio de la zona, 7 días pre...","agregado (zona, fecha)","[-7d, -1d]",no,gold_taxi_v1
9,pasajeros_faltante,flag de calidad,1 si passenger_count venía nulo,regla completitud,—,no,gold_taxi_v1


In [29]:
# Tabla Gold persistida: insumo directo del modelo de la Fase 2 (opcional, ~200 MB).
COLUMNAS_GOLD = (["PULocationID", "DOLocationID", "fecha", "tpep_pickup_datetime"]
                 + NUM_DIRECTAS + A_IMPUTAR + CATEGORICAS + ["propina_alta"])
# gold.select(COLUMNAS_GOLD).write.mode("overwrite").partitionBy("fecha").parquet("gold_taxi_v1")
print("Columnas de la capa Gold:", len(COLUMNAS_GOLD))


Columnas de la capa Gold: 16


## 8. Ejercicios de extensión (Propuestos)

**E1 · Target encoding sin fuga.** Codificar <code>PULocationID</code> (265 categorías: one-hot la infla, StringIndexer le impone un orden falso) por la tasa media de <code>propina_alta</code> de la zona. La media debe calcularse **solo con train** y luego unirse a train y test. Dos preguntas: ¿qué pasa con una zona presente en test y ausente en train, y qué ocurriría si la media se calculara sobre <code>gold</code> completo?

**E2 · Muestreo estratificado.** Rehacer las secciones 4 y 6 sobre una muestra con <code>sampleBy("propina_alta", fractions={0: 0.1, 1: 0.1})</code> y comparar tiempos y la distribución del objetivo. Consigna de la clase: desarrollar con muestra, reportar con el total.

**E3 · <code>rowsBetween</code> vs <code>rangeBetween</code>.** Recalcular <code>demanda_media_7d</code> con <code>rowsBetween(-7, -1)</code> y comparar contra la versión por rango en una zona de baja demanda (p. ej. <code>PULocationID = 5</code>). ¿Cuánto difieren y en qué dirección sesga el error?

**E4 · PCA.** Aplicar <code>PCA(k=10)</code> sobre el one-hot de las 265 zonas y ver cuánta varianza explican los 10 primeros componentes. ¿Vale el costo de perder la interpretabilidad de la zona?

**E5 · Fuga inducida.** Agregar <code>total_amount</code> al <code>VectorAssembler</code>, entrenar una regresión logística y comparar el AUC contra el modelo sin ella. El salto que se observe es la medida exacta de lo que una fuga promete y no cumple en producción.

## Declaración de uso de IA (obligatoria)

Indicar si se usaron asistentes de IA (text-to-SQL, autocompletado, chat): **en qué celdas, con qué prompt y cómo se validó el resultado**. Si no se usaron, escribir "Sin uso de IA".

---
**Entrega:** notebook ejecutado completamente (Kernel → Restart & Run All antes de subir), con las celdas-respuesta de las secciones **1, 2b, 3.4 y 5b** completas (ninguna justificación puede quedar en `PENDIENTE`) y el archivo <code>diccionario_features_v1.csv</code> adjunto.

**Los tres artefactos de este lab son secciones del informe de la Fase 1 (vie 28 ago):** el reporte de calidad con decisiones, el registro de features con su veredicto de fuga, y la justificación del split temporal.